In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import TruncatedSVD
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

In [2]:
df = pd.read_csv('/Users/loohanweiaugustine/Desktop/Final Coursework IDTA/Submission/Csv Files..../Census.csv')

In [3]:
df_classification = df.copy()

In [4]:
df_classification = df_classification[df_classification['Approximated Social Grade'] != -9].copy()

In [5]:
target_classification = 'Approximated Social Grade'
y_clf = df_classification[target_classification]
features_to_drop = [target_classification, 'Person ID', 'Hours worked per week', 'No of hours']
X_clf = df_classification.drop(columns=features_to_drop)

In [6]:
categorical_features_clf = X_clf.select_dtypes(include=['int64', 'object']).columns

In [7]:
preprocessor_clf = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features_clf)
    ],
    remainder='passthrough'
)

In [8]:
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(X_clf, y_clf, test_size=0.2, random_state=23, stratify=y_clf)

In [9]:
print("\nLogistic Regression Classifier")
pipeline_lr = Pipeline(steps=[
    ('preprocessor', preprocessor_clf),
    ('classifier', LogisticRegression(max_iter=10000))
])

param_grid_lr = {
    'classifier__C': [0.01, 0.1, 1, 10],
    'classifier__solver': ['lbfgs']
}

grid_lr = GridSearchCV(pipeline_lr, param_grid_lr, cv=5, scoring='accuracy', n_jobs=-1)
grid_lr.fit(X_train_clf, y_train_clf)

y_pred_lr = grid_lr.predict(X_test_clf)
y_prob_lr = grid_lr.predict_proba(X_test_clf)

print("Best Parameters:", grid_lr.best_params_)
print("Accuracy:", accuracy_score(y_test_clf, y_pred_lr))
print("Confusion Matrix:\n", confusion_matrix(y_test_clf, y_pred_lr))
print("Classification Report:\n", classification_report(y_test_clf, y_pred_lr))
print("AUC-ROC Score:", roc_auc_score(y_test_clf, y_prob_lr, multi_class='ovr'))


Logistic Regression Classifier
Best Parameters: {'classifier__C': 10, 'classifier__solver': 'lbfgs'}
Accuracy: 0.7602997935553362
Confusion Matrix:
 [[12170  3337   257   700]
 [ 3565 25038   942  2384]
 [  224   674 10153  4936]
 [  314   821  3210 20403]]
Classification Report:
               precision    recall  f1-score   support

           1       0.75      0.74      0.74     16464
           2       0.84      0.78      0.81     31929
           3       0.70      0.64      0.66     15987
           4       0.72      0.82      0.77     24748

    accuracy                           0.76     89128
   macro avg       0.75      0.75      0.75     89128
weighted avg       0.76      0.76      0.76     89128

AUC-ROC Score: 0.9229157479595086


In [10]:
print("\nRandom Forest Classifier")
pipeline_rf = Pipeline(steps=[
    ('preprocessor', preprocessor_clf),
    ('classifier', RandomForestClassifier(random_state=23, n_jobs=-1))
])

param_grid_rf = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [10, 20],
    'classifier__min_samples_split': [2, 5]
}

grid_rf = GridSearchCV(pipeline_rf, param_grid_rf, cv=3, scoring='accuracy', n_jobs=-1)
grid_rf.fit(X_train_clf, y_train_clf)

y_pred_rf = grid_rf.predict(X_test_clf)
y_prob_rf = grid_rf.predict_proba(X_test_clf)

print("Best Parameters:", grid_rf.best_params_)
print("Accuracy:", accuracy_score(y_test_clf, y_pred_rf))
print("Confusion Matrix:\n", confusion_matrix(y_test_clf, y_pred_rf))
print("Classification Report:\n", classification_report(y_test_clf, y_pred_rf))
print("AUC-ROC Score:", roc_auc_score(y_test_clf, y_prob_rf, multi_class='ovr'))


Random Forest Classifier


Exception ignored in: <function ResourceTracker.__del__ at 0x111361bc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
/opt/anaconda3/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
Exception ignored in: <function ResourceTracker.__del__ at 0x105561bc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  F

Best Parameters: {'classifier__max_depth': 20, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 50}
Accuracy: 0.7872273584058882
Confusion Matrix:
 [[12156  3368   149   791]
 [ 2538 26532   762  2097]
 [  189   752 10405  4641]
 [  251  1119  2307 21071]]
Classification Report:
               precision    recall  f1-score   support

           1       0.80      0.74      0.77     16464
           2       0.84      0.83      0.83     31929
           3       0.76      0.65      0.70     15987
           4       0.74      0.85      0.79     24748

    accuracy                           0.79     89128
   macro avg       0.78      0.77      0.77     89128
weighted avg       0.79      0.79      0.79     89128

AUC-ROC Score: 0.9509111802357636


In [11]:
print("\nKNearest Neighbour (KNN) Classifier")
pipeline = Pipeline([
    ('preprocessor', preprocessor_clf),
    ('SVD', TruncatedSVD(n_components=50, n_iter=2, random_state=23)),
    ('StandardScaler', StandardScaler(with_mean=True)),
    ('Classifier', KNeighborsClassifier(
        n_neighbors=25,
        weights='uniform',
        algorithm='ball_tree',
        leaf_size=60,
        metric='minkowski', p=2,
        n_jobs=-1
    ))
])

pipeline.fit(X_train_clf, y_train_clf)

y_pred_knn = pipeline.predict(X_test_clf)
y_prob_knn = pipeline.predict_proba(X_test_clf)

print("Accuracy:", accuracy_score(y_test_clf, y_pred_knn))
print("Confusion Matrix:\n", confusion_matrix(y_test_clf, y_pred_knn))
print("Classification Report:\n", classification_report(y_test_clf, y_pred_knn))
print("AUC-ROC (OvR):", roc_auc_score(y_test_clf, y_prob_knn, multi_class='ovr'))


KNearest Neighbour (KNN) Classifier


Exception ignored in: <function ResourceTracker.__del__ at 0x107895bc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x113061bc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x106f61bc0>
Traceback (most recent call last

Accuracy: 0.7692083295933938
Confusion Matrix:
 [[11913  3590   280   681]
 [ 2841 25966   900  2222]
 [  285   964 10381  4357]
 [  386  1258  2806 20298]]
Classification Report:
               precision    recall  f1-score   support

           1       0.77      0.72      0.75     16464
           2       0.82      0.81      0.82     31929
           3       0.72      0.65      0.68     15987
           4       0.74      0.82      0.78     24748

    accuracy                           0.77     89128
   macro avg       0.76      0.75      0.76     89128
weighted avg       0.77      0.77      0.77     89128

AUC-ROC (OvR): 0.9339630744648642


In [12]:
results = {
    'Model': [],
    'Accuracy': [],
    'Precision (Macro Avg)': [],
    'Recall (Macro Avg)': [],
    'F1-score (Macro Avg)': [],
    'AUC-ROC': []
}

def extract_metrics(model_name, y_test, y_pred, y_prob):
    report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    results['Model'].append(model_name)
    results['Accuracy'].append(accuracy_score(y_test, y_pred))
    results['Precision (Macro Avg)'].append(report['macro avg']['precision'])
    results['Recall (Macro Avg)'].append(report['macro avg']['recall'])
    results['F1-score (Macro Avg)'].append(report['macro avg']['f1-score'])
    results['AUC-ROC'].append(roc_auc_score(y_test, y_prob, multi_class='ovr'))

extract_metrics('Logistic Regression (Tuned)', y_test_clf, y_pred_lr, y_prob_lr)
extract_metrics('Random Forest (Tuned)', y_test_clf, y_pred_rf, y_prob_rf)
extract_metrics('KNearest Neighbour (KNN)', y_test_clf, y_pred_knn, y_prob_knn)

results_df = pd.DataFrame(results)
print("\n--- Final Model Comparison Table ---")
print(results_df.to_markdown(index=False))


--- Final Model Comparison Table ---
| Model                       |   Accuracy |   Precision (Macro Avg) |   Recall (Macro Avg) |   F1-score (Macro Avg) |   AUC-ROC |
|:----------------------------|-----------:|------------------------:|---------------------:|-----------------------:|----------:|
| Logistic Regression (Tuned) |   0.7603   |                0.750289 |             0.745719 |               0.746489 |  0.922916 |
| Random Forest (Tuned)       |   0.787227 |                0.784714 |             0.767893 |               0.773798 |  0.950911 |
| KNearest Neighbour (KNN)    |   0.769208 |                0.762135 |             0.751587 |               0.755611 |  0.933963 |
